# Python Polymorphism

> 📘 **Python Mastery** · Module 08 — Object-Oriented Programming (OOP) · Lesson 4/5

*Poly* = many, *morph* = forms: **polymorphism** lets many different object types answer the **same method call**, each in its own way. One line of calling code, many behaviors — which is exactly what makes plugins, payment systems, and scikit-learn's swappable estimators possible.

## 🎯 Learning Objectives

- Call the same method across a mixed collection of objects and explain why it works
- Contrast polymorphic dispatch with type-checking `if/else` chains
- Apply duck typing and explain why no shared parent class is required
- Recognize polymorphism already present in built-ins (`len`, `+`)
- Enforce a method contract using `NotImplementedError` in a base class
- Model a real-world system (payment processing) behind one uniform interface

## 1. One Interface, Many Behaviors

Give sibling classes a method with the **same name**, and callers can treat every object identically. The loop below contains exactly one calling line, yet each animal answers differently — because each class supplies its own implementation.

**Syntax:**

```python
for obj in collection_of_mixed_objects:
    obj.common_method()      # each type runs its OWN version
```

**Example:** our farm speaks in four voices.

In [1]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."


class Dog(Animal):
    def speak(self):
        return f"{self.name}: Woof!"


class Cat(Animal):
    def speak(self):
        return f"{self.name}: Meow."


class Cow(Animal):
    def speak(self):
        return f"{self.name}: Moo!"


farm = [Dog("Rex"), Cat("Manik"), Cow("Lali")]

for animal in farm:
    print(animal.speak())      # SAME line of code -- THREE behaviors

Rex: Woof!
Manik: Meow.
Lali: Moo!


In [2]:
# Add a brand-new species: the loop needs ZERO edits. That is the payoff.
class Duck(Animal):
    def speak(self):
        return f"{self.name}: Quack!"


farm.append(Duck("Khaki"))
for animal in farm:
    print(animal.speak())

Rex: Woof!
Manik: Meow.
Lali: Moo!
Khaki: Quack!


## 2. Why Not Just `if/else`?

The procedural alternative branches on each object's type by hand. Every new class then forces edits in *every* branch site scattered around the codebase. Polymorphism moves that decision into the object itself — new types plug in without touching existing code (the *open/closed* idea).

**Example:** the road not taken.

In [3]:
class Dog:
    pass

class Cat:
    pass


def make_sound(animal):
    kind = type(animal).__name__
    if kind == "Dog":
        return "Woof!"
    elif kind == "Cat":
        return "Meow."
    else:
        raise ValueError(f"I don't know a {kind}")


print(make_sound(Dog()))
print(make_sound(Cat()))

# Every new species = hunt down EVERY branch like this, everywhere.
# Polymorphism: give each species a speak() and let objects decide.

Woof!
Meow.


## 3. Duck Typing: 🔍 No Shared Parent Required

*"If it walks like a duck and quacks like a duck, it's a duck."* Python never checks declared types — it only cares whether the object **has the method** when the call happens. Classes with no common ancestor whatsoever can participate in the same loop.

> 🔍 **Under the Hood:** Method calls are resolved at *runtime*: Python looks the attribute name up on the object's type, finds whatever function is there, and calls it. There is no compile-time type checking and no interface declaration to satisfy — the "interface" exists purely as a naming convention between author and caller. This dynamic dispatch is also how built-ins achieve their own polymorphism through special protocols like `__len__` and `__add__`.

**Syntax:**

```python
class A:                 # unrelated to B in every way
    def quack(self): ...

class B:
    def quack(self): ...

for thing in (A(), B()):
    thing.quack()        # works: both simply HAVE quack()
```

**Example:** three strangers, one contract.

In [4]:
# Three classes with NO common parent -- and the loop cannot tell, nor care
class Duck:
    def __init__(self, name):
        self.name = name
    def quack(self):
        return f"{self.name}: Quack!"

class Robot:
    def __init__(self, name):
        self.name = name
    def quack(self):
        return f"{self.name}: Beep-quack module online."

class Prankster:
    def __init__(self, name):
        self.name = name
    def quack(self):
        return f"{self.name}: (it's a human with a reed)"


for candidate in [Duck("Khaki"), Robot("R-2"), Prankster("Rafi")]:
    print(candidate.quack())     # if it quacks like a duck...

Khaki: Quack!
R-2: Beep-quack module online.
Rafi: (it's a human with a reed)


In [5]:
class SilentStone:
    pass                         # no quack() anywhere


try:
    SilentStone().quack()
except AttributeError as err:
    print("AttributeError:", err)

# Duck typing trusts objects to honor the interface;
# Python checks at CALL time, never at class-definition time.

AttributeError: 'SilentStone' object has no attribute 'quack'


## 4. Built-ins Are Polymorphic Too: `len()`

You have used polymorphism since week one. `len()` accepts strings, lists, dicts, ranges — one function name, many implementations, because each type supplies its own `__len__` special method underneath.

**Syntax:**

```python
len(x)        # internally dispatches to type(x).__len__()
```

**Example:**

In [6]:
print(len("Dhaka"))                    # str: counts characters
print(len([10, 20, 30]))               # list: counts elements
print(len({"a": 1, "b": 2, "c": 3}))   # dict: counts keys
print(len(range(100)))                 # range: length without building anything

# One function, four implementations -- each type carries its own __len__:
print("Dhaka".__len__(), [10, 20, 30].__len__())

5
3
3
100
5 3


## 5. Operators Are Polymorphic: Overloading `+`

The `+` operator happily adds ints, concatenates strings, and merges lists — three totally different operations behind one symbol. Your classes join the party by defining `__add__`; this is **operator overloading** (a full tour arrives in Intermediate Python — here is the teaser).

**Syntax:**

```python
class Money:
    def __add__(self, other):        # teaches '+' what Money + Money means
        return Money(self.amount + other.amount)
```

**Example:**

In [7]:
print(7 + 5)                 # ints: arithmetic
print("poly" + "morphism")   # strs: concatenation
print([1, 2] + [3])          # lists: merging
print(3 * "ab")              # bonus: '*' repeats sequences -- polymorphic too

12
polymorphism
[1, 2, 3]
ababab


In [8]:
class Money:
    """An amount in BDT; knows how to add itself to other Money."""

    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):                 # '+' now understands Money
        return Money(self.amount + other.amount)

    def __repr__(self):
        return f"Money({self.amount})"


lunch = Money(250)
coffee = Money(80)
print(lunch + coffee)                         # operator overloading!

Money(330)


## 6. Enforcing the Contract: `NotImplementedError`

Duck typing is trust-based. To make the contract explicit, define the required method in a base class whose body **raises `NotImplementedError`** — subclasses must override it or crash loudly when called. This is the informal version of an abstract base class; the formal machinery looks like:

```python
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self): ...          # Python then refuses Shape() outright
```

Stick with the simple `raise` version until you need enforcement at *instantiation* time rather than call time.

**Syntax:**

```python
class Base:
    def required(self):
        raise NotImplementedError("subclasses must implement required()")
```

**Example:**

In [9]:
class Shape:
    """Base contract: every shape MUST know its area."""

    def area(self):
        raise NotImplementedError("subclass must implement area()")


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return round(3.14159 * self.radius ** 2, 2)


class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2


for s in [Circle(3), Square(4)]:
    print(type(s).__name__, "area =", s.area())

try:
    Shape().area()                             # the contract, left unfilled
except NotImplementedError as err:
    print("NotImplementedError:", err)

Circle area = 28.27
Square area = 16
NotImplementedError: subclass must implement area()


In [10]:
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height


shapes = [Circle(1), Square(2), Triangle(3, 4)]
for s in shapes:
    print(f"{type(s).__name__:>8} -> {s.area():.2f}")   # uniform handling forever

  Circle -> 3.14
  Square -> 4.00
Triangle -> 6.00


## 7. Real World: Uniform Payment Processing

Here is the shape you will meet in production code: several payment providers, each implementing `.pay(amount)`, processed by checkout logic that neither knows nor cares which provider it is holding. Swap providers, add providers — `checkout` never changes.

**Syntax:**

```python
class AnyProvider:
    def pay(self, amount):        # the ONE method everyone must provide
        ...

def checkout(total, methods):
    for m in methods:
        m.pay(total)              # uniform interface
```

**Example:** card, PayPal, and a mobile wallet behind one counter.

In [11]:
class CardPayment:
    def __init__(self, last4):
        self.last4 = last4

    def pay(self, amount):
        return f"Card ****{self.last4}: paid {amount} BDT"


class PayPalPayment:
    def __init__(self, email):
        self.email = email

    def pay(self, amount):
        return f"PayPal <{self.email}>: paid {amount} BDT"


class WalletPayment:                       # bKash/Nagad-style mobile wallet
    def __init__(self, provider):
        self.provider = provider

    def pay(self, amount):
        return f"{self.provider} wallet: paid {amount} BDT"


def checkout(total, methods):
    """Process one total through EVERY payment method uniformly."""
    print(f"-- invoice: {total} BDT --")
    for method in methods:                 # duck typing in production clothes
        print(" ", method.pay(total))


checkout(
    3499,
    [CardPayment("4242"), PayPalPayment("sarah@example.com"), WalletPayment("bKash")],
)

-- invoice: 3499 BDT --
  Card ****4242: paid 3499 BDT
  PayPal <sarah@example.com>: paid 3499 BDT
  bKash wallet: paid 3499 BDT


In [12]:
# A brand-new provider drops in with zero changes to checkout():
class CryptoPayment:
    def pay(self, amount):
        return f"Crypto: paid {amount} BDT (gas fee waived today)"


checkout(199, [CryptoPayment()])

-- invoice: 199 BDT --
  Crypto: paid 199 BDT (gas fee waived today)


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Branching on `type(x).__name__` everywhere | Reimplements polymorphism badly; every new class edits every branch | Call the method; let dispatch decide |
| Trusting duck typing blindly | `AttributeError` erupts mid-loop at runtime | Document the contract; enforce via `raise NotImplementedError` in a base class (or `abc`) |
| Changing the signature when overriding (`speak(self, volume)`) | Existing callers break | Keep signatures compatible; extend with defaulted parameters |
| Confusing `return NotImplemented` with `raise NotImplementedError` | Different tools entirely | Raise `NotImplementedError` for missing methods; **return** the sentinel `NotImplemented` inside dunder operators like `__add__` |
| Building deep hierarchies just to share one method name | Rigid structure for a tiny benefit | Plain duck typing or composition often suffices |

## 💡 Best Practices & Pro Tips

- Program to interfaces, not implementations: name methods by intent (`pay`, `area`, `speak`) and keep those names stable.
- Aim for open/closed design — adding a new class should require zero edits to working loops.
- Keep overrides signature-compatible so subclasses remain drop-in replacements.
- Implement the built-in protocols (`__len__`, `__iter__`, `__lt__`, ...) and your objects instantly play with `len()`, `for` loops, and `sorted()` — free polymorphism.
- 🤖 **AI-engineering relevance:** entire ML ecosystems are polymorphic interfaces — `model.fit(X, y)` / `model.predict(X)` let you swap a Random Forest for a neural net in one line; loss functions, activations, tokenizers, and data collators are all interchangeable behind a single method signature. Writing a custom layer or autograd function *is* overriding contract methods.

## 📌 Summary

| Mechanism | What it does | Example |
|---|---|---|
| Overriding + a loop | One call site, many behaviors | `for a in farm: a.speak()` |
| Duck typing | Interface by convention, no shared parent needed | `thing.quack()` |
| `len(obj)` | Dispatches to the type's `__len__` | `len([1, 2])` |
| `+` operator | Dispatches to `__add__` | `lunch + coffee` |
| `raise NotImplementedError(...)` | Informal abstract contract in a base class | `Shape.area()` |
| `from abc import ABC, abstractmethod` | Formal contracts (preview) | see snippet above |

**Key takeaways**
- Polymorphism = same call, per-type behavior; callers stop caring about concrete types.
- Python checks methods at call time — duck typing needs no shared ancestor.
- Built-ins (`len`, `+`, `sorted`) are proof this pattern scales to the whole language.
- Contracts turn polite convention into loud failure, keeping mixed collections safe.

> 🔗 **Next Lesson:** [05 · Python Encapsulation](../05_Encapsulation/) — guard an object's internals so its guarantees actually hold. Our banking example gets its final, validated form.